In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp lặp đơn (Fixed-Point Interation)

Điều kiện hội tụ:

* $(a, b)$ là khoảng cách ly nghiệm của phương trình. 

* $|\varphi'(x)| \leq q < 1 \ \forall x \in [a, b]$ --> đảm bảo nghiệm duy nhất

Ánh xạ co không thoả mãn điều kiện --> tự co lại ánh xạ đến khi thoả mãn

# Thuật toán tổng quát

1. Tạo phương trình lặp $x = \varphi(x)$ tương đương với phương trình $f(x) = 0$.

2. Công thức lặp tương ứng với phương trình lặp: $x_{k+1} = \varphi(x_k)$

3. Nếu $\exists \lim\limits_{k \to \infty} x_k = \alpha$ và $\varphi(x)$ liên tục thì $\alpha = \varphi(\alpha)$.


# Áp dụng

Note: Chọn các hàm $\varphi$ ổn định trong tính toán (chọn sao cho độ dốc càng nhỏ càng tốt)

## 1. Tiên nghiệm: 
Đánh giá sai số: $|x_n - x^*| \leq \dfrac{q^n}{1 - q} |x_1 - x_0| < \varepsilon$



### 1.1 Thuật toán
#### Input:
- Khoảng cách ly nghiệm $(a,b)$
- Sai số $\varepsilon$
- Hàm lặp $\varphi(x)$ ứng với phương trình lặp $x=\varphi(x)$ tương đương với phương trình $f(x)=0$
#### Output: nghiệm gần đúng $x$
#### Thuật toán:
- Bước 1: Tính $q=\max\limits_{[a,b]}|\varphi'(x)|$. Nếu $q \geq 1$: dừng chương trình do không thoả mãn điều kiện hội tụ.

- Bước 2: Chọn $x_0$ bất kỳ thuộc $[a,b]$. Tính $x_1 = \varphi(x_0)$

- Bước 3: Tính số lần lặp: $n = \left[\log_q \left(\varepsilon \cdot \dfrac{1-q}{|x_1 - x_0|} \right)\right]+1$
- Bước 4: Khởi tạo $i=0$. Khi $i \leq n $:
    - Tính $x_{i+1} = \varphi(x_i)$
    - Nếu $f(x_{i+1}) = 0$: thu được nghiệm $x=x_{i+1}$, dừng chương trình
    - Cập nhật $i=i+1$
- Bước 5: Sau khi kết thúc $n$ vòng lặp, thu được nghiệm gần đúng là $x=\varphi(x_n)$. Dừng chương trình.

### 1.2 Code
#### 1.2.1 Có cộng thêm sai số làm tròn

In [2]:
def fixed_point_iteration_v1 (f, phi, dphi, a, b, x_0, q, n, rbl):
    #Initial value
    x = x_0
    results = []
    results.append({
        'n': 0,
        'x_n': x_0,
        'f(x_n)': f(x_0),
        'delta=q^n/(1-q) * |x_1-x_0|': None
        }) 
    
    # Fixed-point iteration
    delta = 0 
    diff = abs(phi(x) - x)
    temp_q = q
    for i in range(n):
        # Compute next iteration
        x_new = phi(x)

        delta = temp_q/(1-q) * diff
        temp_q *= q
        
        results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'delta=q^n/(1-q) * |x_1-x_0|': delta
        }) 

        # Prepare for next iteration
        x = x_new
        if (f(x_new) == 0): 
            break

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    if rbl == None:
        print(f"The value of root is: {x}")
    else:
        total_delta = delta + 0.5 * 10**(-rbl)
        print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
        print(f"Relative error is: {total_delta}")

In [4]:
f = lambda x: x**5 - 17*x + 2
phi = lambda x: (x**5 + 2) / 17
dphi = lambda x: 5*x**4 / 17

a = 0
b = 1
x_0 = a
q = max(np.abs(dphi(x)) for x in [a, b]) # Lipschitz constant (q<1)

n = 15
rbl = 9

fixed_point_iteration_v1 (f, phi, dphi, a, b, x_0, q, n, rbl)

 n               x_n            f(x_n)  delta=q^n/(1-q) * |x_1-x_0|
 0 0.000000000000000 2.000000000000000                          NaN
 1 0.117647058823529 0.000022537480887            0.049019607843137
 2 0.117648384557699 0.000000001269874            0.014417531718570
 3 0.117648384632398 0.000000000000071            0.004240450505462
 4 0.117648384632402 0.000000000000000            0.001247191325136
The value of root with 9 decimal point is: 0.117648385
Relative error is: 0.0012471918251357945


#### Sai số tương đối

Từ công thức sai số tiên nghiệm, suy ra Điều kiện dừng: $n > \log_q \left(\varepsilon \cdot \dfrac{1-q}{|x_1 - x_0|} \right)$

## 2. Hậu nghiệm - Sai số tuyệt đối:

Đánh giá sai số dựa trên công thức: $|x_n - x^*| \leq \dfrac{q}{1 - q} |x_n - x_{n-1}| < \varepsilon$.
### 2.1 Thuật toán
#### Input:
- Khoảng cách ly nghiệm $(a,b)$
- Sai số $\varepsilon$
- Hàm lặp $\varphi(x)$ ứng với phương trình lặp $x=\varphi(x)$ tương đương với phương trình $f(x)=0$
#### Output: nghiệm gần đúng $x$
#### Thuật toán:
- Bước 1: Tính $q=\max\limits_{[a,b]}|\varphi'(x)|$. Nếu $q \geq 1$: dừng chương trình do không thoả mãn điều kiện hội tụ.

- Bước 2: Chọn $x_0$ bất kỳ thuộc $[a,b]$

- Bước 3: Khởi tạo $i=0$. Lặp đến khi thoả mãn điều kiện dừng $|x_n-x_{n-1}|<\dfrac{(1-q)\varepsilon}{q}$:
    - Tính $x_{i+1} = \varphi(x_i), \quad \Delta x_i =|x_{i+1}-x_i|$
    - Nếu $f(x_{i+1}) = 0$: thu được nghiệm $x=x_{i+1}$, dừng chương trình
    - Nếu thoả mãn điều kiện dừng $\Delta x_i < \dfrac{(1-q)\varepsilon}{q}$: dừng chương trình. Trái lại cập nhật $i=i+1$
- Bước 4: Sau khi thoả mãn điều kiện dừng với $n$ lần lặp, thu được nghiệm gần đúng là $x=\varphi(x_{n-1})$. Dừng chương trình.

### 2.2 Code
#### 2.2.1 Có cộng thêm sai số làm tròn

In [ ]:
def fixed_point_iteration_v2 (f, phi, dphi, a, b, x_0, q, n, rbl):
    #Initial value
    x = x_0
    results = []
    results.append({
        'n': 0,
        'x_n': x_0,
        'f(x_n)': f(x_0),
        'delta_x = q/(1-q) * |x_n - x|': None
        }) 
    
    # Fixed-point iteration
    delta_x = 0
    for i in range(n):
        # Compute next iteration
        x_new = phi(x)
        delta_x = q/(1-q) * abs(x_new - x)
        
        results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'delta_x = q/(1-q) * |x_n - x|': delta_x
        }) 

        # Prepare for next iteration
        x = x_new
        if (f(x_new) == 0): 
            break

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    if rbl == None:
        print(f"The value of root is: {x}")
    else:
        total_delta = delta_x + 0.5 * 10**(-rbl)
        print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
        print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x**5 - 17*x + 2
phi = lambda x: (x**5 + 2) / 17
dphi = lambda x: 5*x**4 / 17

a = 0
b = 1
x_0 = a
q = max(np.abs(dphi(x)) for x in [a, b]) # Lipschitz constant (q<1)

n = 15
rbl = 9

fixed_point_iteration_v2 (f, phi, dphi, a, b, x_0, q, n, rbl)

 n               x_n            f(x_n)  delta_x = q/(1-q) * |x_n - x|
 0 0.000000000000000 2.000000000000000                            NaN
 1 0.117647058823529 0.000022537480887              0.049019607843137
 2 0.117648384557699 0.000000001269874              0.000000552389237
 3 0.117648384632398 0.000000000000071              0.000000000031124
 4 0.117648384632402 0.000000000000000              0.000000000000002
The value of root with 9 decimal point is: 0.117648385
Relative error is: 5.000017520707108e-10


#### 2.2.2 Không cộng thêm sai số làm tròn

Từ công thức sai số hậu nghiệm, ta có điều kiện dừng: $|x_n - x_{n-1}| < \dfrac{1-q}{q} \varepsilon $

In [3]:
def fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps):
    #Initial value
    x = x_0
    results = []
    # Fixed-point iteration
    i=0
    new_eps = (1-q)/q * eps
    print(f"new_eps = {new_eps}")
    
    while True:
        # Compute next iteration
        x_new = phi(x)
        current_delta_x = abs(x_new - x)
        
        results.append({
            'n': i,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=|x_(n+1) - x_n|': current_delta_x
        }) 

        x = x_new  
        #stop condition 
        if (f(x_new) == 0): 
            break
        elif current_delta_x < new_eps:
            break
        else:
            i += 1

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    print(f"The value of root with absolute error {eps} is: {x}")

Câu 13: Dùng phương pháp lặp đơn giải gần đúng các phương trình dưới đây với 8 chữ số đáng tin:

a. $x^3+3x^2-1=0$

Nghiệm $x_1$:

In [28]:
f = lambda x: x**3 + 3*x**2 - 1
phi = lambda x: np.cbrt(1-3*x**2)
dphi = lambda x: -2*x*((np.cbrt(1-3*x**2))/(1-3*x**2))

a = -3
b = -2
x_0 = -3
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-7)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 1.1826093040608754e-08
 n            x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 -2.962496068407370 -0.670851134012615        0.037503931592630
 1 -2.936794324180405 -0.454866158372653        0.025701744226965
 2 -2.919108171521692 -0.310705156480577        0.017686152658713
 3 -2.906902998616905 -0.213322421030306        0.012205172904787
 4 -2.898463519421713 -0.146982809848367        0.008439479195193
 5 -2.892619851879761 -0.101523497788829        0.005843667541952
 6 -2.888569704576636 -0.070244007872809        0.004050147303125
 7 -2.885760749995907 -0.048659495943024        0.002808954580728
 8 -2.883811717586956 -0.033735251175344        0.001949032408952
 9 -2.882458920583316 -0.023401780924470        0.001352797003640
10 -2.881519753394323 -0.016240018945958        0.000939167188993
11 -2.880867644141627 -0.011273118418639        0.000652109252695
12 -2.880414804466182 -0.007826811822529        0.000452839675445
13 -2.880100318705442 -0.005434799942300   

Nghiệm $x_2$:

In [ ]:
f = lambda x: x**3 + 3*x**2 - 1
phi = lambda x: -np.sqrt(1/(x+3))
dphi = lambda x: 1/(2*np.sqrt(x+3)*(x+3))

a = -1
b = -0.5
x_0 = -0.5
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-8)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 2.3284271247461902e-08
 n            x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 -0.632455532033676 -0.052982212813470        0.132455532033676
 1 -0.649906570390294 -0.007370944281189        0.017451038356618
 2 -0.652315106044928 -0.001024868043240        0.002408535654634
 3 -0.652649631653508 -0.000142491698712        0.000334525608580
 4 -0.652696135200683 -0.000019811080528        0.000046503547175
 5 -0.652702600604596 -0.000002754395802        0.000006465403913
 6 -0.652703499507104 -0.000000382952117        0.000000898902509
 7 -0.652703624484234 -0.000000053243009        0.000000124977129
 8 -0.652703641860187 -0.000000007402539        0.000000017375953
The value of root with absolute error 5e-09 is: -0.652703641860187


Nghiệm $x_3$:

In [17]:
f = lambda x: x**3 + 3*x**2 - 1
phi = lambda x: np.sqrt(1/(x+3))
dphi = lambda x: -1/(2*np.sqrt(x+3)*(x+3))

a = 0.5
b = 1  
x_0 = 0.5
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-8)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 6.047900426854397e-08
 n           x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 0.534522483824849  0.009863566807100        0.034522483824849
 1 0.531905677161984 -0.000740356490824        0.002616806662865
 2 0.532102686471873  0.000055779889923        0.000197009309889
 3 0.532087846778048 -0.000004201376671        0.000014839693825
 4 0.532088964532303  0.000000316457094        0.000001117754255
 5 0.532088880340659 -0.000000023836218        0.000000084191644
 6 0.532088886682153  0.000000001795395        0.000000006341494
The value of root with absolute error 5e-09 is: 0.532088886682153


b. $x^2+4\sin x -1=0$

Nghiệm $x_1$:

In [30]:
f = lambda x: x**2 + 4*np.sin(x) - 1
phi = lambda x: -np.sqrt(1-4*np.sin(x))
dphi = lambda x: (2*np.cos(x))/(np.sqrt(1-4*np.sin(x)))

a = -2.5
b = -2
x_0 = -2
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-7)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 7.488161858603952e-09
 n            x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 -2.153413501235359  0.297087656919069        0.153413501235359
 1 -2.083291158331849 -0.145993441048605        0.070122342903510
 2 -2.118040483898328  0.070248584216216        0.034749325566479
 3 -2.101391659642735 -0.034176920905103        0.016648824255593
 4 -2.109507958771702  0.016542365321425        0.008116299128967
 5 -2.105583402005184 -0.008027207476505        0.003924556766517
 6 -2.107488711779076  0.003890468712188        0.001905309773892
 7 -2.106565498996896 -0.001886675555473        0.000923212782180
 8 -2.107013259834763  0.000914677019867        0.000447760837867
 9 -2.106796193299116 -0.000443505350151        0.000217066535646
10 -2.106901446544142  0.000215030772775        0.000105253245026
11 -2.106850415828571 -0.000104259730646        0.000051030715571
12 -2.106875158714362  0.000050550526425        0.000024742885791
13 -2.106863162115955 -0.000024509706565    

Nghiệm $x_2$:

In [12]:
f = lambda x: x**2 + 4*np.sin(x) - 1
phi = lambda x: np.arcsin((1-x**2)/4)
dphi = lambda x: (-2*x)/(np.sqrt(-x**4+2*x**2+15))

a = 0.1
b = 1
x_0 = 0.1
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-8)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 5e-09
 n           x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 0.250099123469688  0.052549571560306        0.150099123469688
 1 0.236562862896508 -0.006587583458514        0.013536260573179
 2 0.238257286962825  0.000804546689094        0.001694424066316
 3 0.238050308353374 -0.000098585483749        0.000206978609450
 4 0.238075670009632  0.000012075343398        0.000025361656257
 5 0.238072563552943 -0.000001479133865        0.000003106456689
 6 0.238072944069141  0.000000181181079        0.000000380516199
 7 0.238072897459203 -0.000000022193128        0.000000046609938
 8 0.238072903168521  0.000000002718468        0.000000005709318
 9 0.238072902469178 -0.000000000332989        0.000000000699342
The value of root with absolute error 5e-09 is: 0.23807290246917837


c. $(1.4)^x-x=0$

Nghiệm $x_1$:

In [38]:
f = lambda x: (1.4)**x - x
phi = lambda x: (1.4)**x
dphi = lambda x: (1.4)**x*np.log(1.4)

a = 1
b = 2
x_0 = 2
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-7)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 2.5816668673175056e-08
 n           x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 1.960000000000000 -0.026202698209403        0.040000000000000
 1 1.933797301790597 -0.016974349187362        0.026202698209403
 2 1.916822952603235 -0.010916533309999        0.016974349187362
 3 1.905906419293236 -0.006987763380521        0.010916533309999
 4 1.898918655912715 -0.004459470883830        0.006987763380521
 5 1.894459185028885 -0.002840481955769        0.004459470883830
 6 1.891618703073116 -0.001807038263358        0.002840481955769
 7 1.889811664809758 -0.001148690650563        0.001807038263358
 8 1.888662974159196 -0.000729831934529        0.001148690650563
 9 1.887933142224666 -0.000463559392009        0.000729831934529
10 1.887469582832657 -0.000294374856061        0.000463559392009
11 1.887175207976596 -0.000186913496486        0.000294374856061
12 1.886988294480110 -0.000118671230683        0.000186913496486
13 1.886869623249427 -0.000075340396859        0.00011867

Nghiệm $x_2$:

In [42]:
f = lambda x: (1.4)**x - x
phi = lambda x: np.emath.logn(1.4,x)
dphi = lambda x: 1/(x*np.log(1.4))

a = 4
b = 5
x_0 = 9/2
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-7)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 1.7294447324242584e-08
 n           x_(n+1)        f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 4.470138195887778 0.029861804112222        0.029861804112222
 1 4.450350315646709 0.019787880241068        0.019787880241069
 2 4.437164952119557 0.013185363527152        0.013185363527152
 3 4.428346490532821 0.008818461586735        0.008818461586736
 4 4.422434007102590 0.005912483430230        0.005912483430230
 5 4.418463287758387 0.003970719344203        0.003970719344204
 6 4.415793642009517 0.002669645748869        0.002669645748870
 7 4.413997401923793 0.001796240085723        0.001796240085724
 8 4.412788211429249 0.001209190494542        0.001209190494544
 9 4.411973933039055 0.000814278390193        0.000814278390194
10 4.411425465836428 0.000548467202627        0.000548467202627
11 4.411055981965042 0.000369483871387        0.000369483871387
12 4.410807047234283 0.000248934730759        0.000248934730759
13 4.410639319089724 0.000167728144559        0.000167728144559
14 4.41

d. $e^x-10x+7=0$

Nghiệm $x_1$:

In [46]:
f = lambda x: np.exp(x) - 10*x + 7
phi = lambda x: (np.exp(x)+7)/10
dphi = lambda x: (np.exp(x))/10

a = 0.5
b = 1
x_0 = 1
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-8)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 1.339397205857212e-08
 n           x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 0.971828182845904 -0.075510313269952        0.028171817154096
 1 0.964277151518909 -0.019880496911582        0.007551031326995
 2 0.962289101827751 -0.005209257831385        0.001988049691158
 3 0.961768176044613 -0.001363262810866        0.000520925783139
 4 0.961631849763526 -0.000356648660075        0.000136326281087
 5 0.961596184897518 -0.000093296265115        0.000035664866007
 6 0.961586855271007 -0.000024404962756        0.000009329626512
 7 0.961584414774731 -0.000006383950110        0.000002440496276
 8 0.961583776379720 -0.000001669937244        0.000000638395011
 9 0.961583609385996 -0.000000436828175        0.000000166993725
10 0.961583565703178 -0.000000114267069        0.000000043682817
11 0.961583554276471 -0.000000029890387        0.000000011426707
The value of root with absolute error 5e-09 is: 0.9615835542764714


Nghiệm $x_2$:

In [8]:
f = lambda x: np.exp(x) - 10*x + 7
phi = lambda x: np.log(10*x-7)
dphi = lambda x: 10/(10*x-7)
a = 3
b = 4
x_0 = 3
q = max(np.abs(dphi(x)) for x in [a, b])

eps = 0.5 * 10**(-7)

fixed_point_recursion_absolute (f, phi, dphi, a, b, x_0, q, eps)

new_eps = 6.500000000000001e-08
 n           x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 3.135494215929150 -1.354942159291497        0.135494215929150
 1 3.192734792354905 -0.572405764257560        0.057240576425756
 2 3.215965510951988 -0.232307185970825        0.023230718597083
 3 3.225241723849569 -0.092762128975810        0.009276212897581
 4 3.228921883356984 -0.036801595074142        0.003680159507415
 5 3.230378171852499 -0.014562884955158        0.001456288495515
 6 3.230953859603707 -0.005756877512081        0.000575687751208
 7 3.231181344285650 -0.002274846819429        0.000227484681942
 8 3.231271221253741 -0.000898769680909        0.000089876968091
 9 3.231306728536796 -0.000355072830558        0.000035507283056
10 3.231320755889425 -0.000140273526288        0.000014027352629
11 3.231326297420052 -0.000055415306274        0.000005541530627
12 3.231328486603129 -0.000021891830773        0.000002189183077
13 3.231329351439137 -0.000008648360083        0.000000864

#### 2.2.3 Tìm nghiệm thoả mãn điều kiện sai số tương đối

Đánh giá sai số dựa trên công thức: $\dfrac{|x_n - x^*|}{|x_n|} \leq \dfrac{q}{1 - q} \cdot \dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta$.  

#### Công thức mục tiêu:

Từ công thức sai số Hậu nghiệm, suy ra Điều kiện dừng: $\dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta \cdot \dfrac{1-q}{q}$

In [19]:
def fixed_point_recursion_relative (f, phi, dphi, a, b, x_0, q, eta):
    #Initial value
    x = x_0; results = [];
    results.append({
        'n': 0,
        'x_n': x_0,
        'f(x_n)': f(x_0),
        'sigma_x=|x_n-x_(n-1)| / |x_n|': None
        }) 
    
    # Fixed-point iteration
    i=0; new_eta = (1-q)/q * eta
    print(f"new_eta = {new_eta}")
    
    while True:
        # Compute next iteration
        x_new = phi(x)
        current_sigma_x = abs(x_new - x) / abs(x_new)
        
        results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'sigma_x=|x_n-x_(n-1)| / |x_n|': current_sigma_x
        }) 

        #stop condition 
        if (f(x) == 0): 
            break
        elif current_sigma_x < new_eta:
            break
        else:
            i += 1
            x = x_new  

	#Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))
		
    print(f"The value of root with relative error {eps} is: {x}")

In [20]:
f = lambda x: x**2 + 3*x - 1
phi = lambda x: 1/(x+3)
dphi = lambda x: -1/(x+3)**2

a = 0
b = 1
x_0 = a
q = max(np.abs(dphi(x)) for x in [a, b]) # Lipschitz constant (q<1)

eta = 5 * 10**(-9)

fixed_point_recursion_relative (f, phi, dphi, a, b, x_0, q, eta)

new_eta = 4e-08
 n               x_n             f(x_n)  sigma_x=|x_n-x_(n-1)| / |x_n|
 0 0.000000000000000 -1.000000000000000                            NaN
 1 0.333333333333333  0.111111111111111              1.000000000000000
 2 0.300000000000000 -0.010000000000000              0.111111111111111
 3 0.303030303030303  0.000918273645546              0.010000000000000
 4 0.302752293577982 -0.000084167999327              0.000918273645547
 5 0.302777777777778  0.000007716049383              0.000084167999327
 6 0.302775441547519 -0.000000707353148              0.000007716049383
 7 0.302775655716832  0.000000064845254              0.000000707353148
 8 0.302775636083269 -0.000000005944565              0.000000064845254
 9 0.302775637883138  0.000000000544957              0.000000005944565
The value of root with relative error 1e-08 is: 0.3027756360832691
